# GNN-based P&ID Digital Twin

Builds graph-based features and demonstrates GraphSAGE/GNN workflow for reactor/equipment node representation.


In [ ]:
import numpy as np

def find_tag_xy(df_text, tag):
    pat = rf"\b{tag.replace('-','[-\s]?')}\b"
    hits = df_text[df_text["text"].astype(str).str.contains(pat, case=False, regex=True)]
    if len(hits) == 0:
        raise ValueError(f"Tag not found: {tag}")
    return hits.iloc[0][["x","y"]].values.astype(float)

def nearest_insert_index(df_main, xy):
    pts = df_main[["x","y"]].values.astype(float)
    d = np.sqrt(((pts - xy)**2).sum(axis=1))
    return df_main.iloc[np.argmin(d)].name

reactor_tags = ["R-100","R-200","R-300"]
reactor_nodes = {}

for t in reactor_tags:
    xy = find_tag_xy(df_text, t)
    ins_idx = nearest_insert_index(df_main, xy)
    reactor_nodes[t] = f"ins_{ins_idx}"
    # stamp role
    G.nodes[reactor_nodes[t]]["unit_tag"] = t
    G.nodes[reactor_nodes[t]]["unit_type"] = "reactor"

G.nodes[reactor_nodes["R-300"]]["reactor_stage"] = "catalytic_fixed_bed"
G.nodes[reactor_nodes["R-100"]]["reactor_stage"] = "acidification"
G.nodes[reactor_nodes["R-200"]]["reactor_stage"] = "acidification"

reactor_nodes


In [ ]:
import torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import SAGEConv

# --- index mapping
node_list = list(G.nodes())
idx = {n:i for i,n in enumerate(node_list)}

r100 = idx[reactor_nodes["R-100"]]
r200 = idx[reactor_nodes["R-200"]]
r300 = idx[reactor_nodes["R-300"]]

def block_hash(s, k=16):
    h = abs(hash(s)) % (2**31)
    v = np.zeros(k, dtype=np.float32)
    v[h % k] = 1.0
    return v

# base node features: (x,y) + hashed block name
base = []
for n in node_list:
    base.append(np.concatenate([
        np.array([G.nodes[n]["x"], G.nodes[n]["y"]], np.float32),
        block_hash(G.nodes[n].get("block","UNK"), 16)
    ]))
base = torch.tensor(np.vstack(base), dtype=torch.float32)

edges = [[idx[u], idx[v]] for u,v in G.edges()]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t().contiguous())
data_static = Data(x=base, edge_index=edge_index)

# Dynamic inputs (example):
# R100: [T,P,acid_dose,tau]
# R200: [T,P,acid_dose,tau]
# R300: [T,P,WHSV,H2_feed,catalyst_family_id,activity]
# catalyst_family_id: encode as 0..N (NiMo,CoMo,Ni2P,Ru,...)
DYN_DIM = 4 + 4 + 6

def make_graph_sample(x_dyn):
    dyn = torch.zeros((data_static.num_nodes, DYN_DIM), dtype=torch.float32)
    dyn[r100, 0:4] = torch.tensor(x_dyn[0:4])
    dyn[r200, 4:8] = torch.tensor(x_dyn[4:8])
    dyn[r300, 8:14] = torch.tensor(x_dyn[8:14])
    return Data(x=torch.cat([data_static.x, dyn], dim=1), edge_index=data_static.edge_index)

class BioOilKPI_GNN(nn.Module):
    def __init__(self, in_dim, hid=96, out_dim=5):
        super().__init__()
        self.c1 = SAGEConv(in_dim, hid)
        self.c2 = SAGEConv(hid, hid)
        self.c3 = SAGEConv(hid, hid)
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, out_dim))
    def forward(self, g):
        h = F.relu(self.c1(g.x, g.edge_index))
        h = F.relu(self.c2(h, g.edge_index))
        h = F.relu(self.c3(h, g.edge_index))
        hg = h.mean(dim=0)               # plant-level KPI readout
        return self.head(hg)

model = BioOilKPI_GNN(in_dim=data_static.x.shape[1]+DYN_DIM, hid=96, out_dim=5)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
